In [2]:
import pandas as pd

In [3]:
pd.options.mode.chained_assignment = None
pd.set_option('display.max_columns', 60)

In [4]:
steam = pd.read_csv('games5.csv')
api = pd.read_csv('api_FINAL_FINAL.csv')

После считывания файлов, переназовем колонки, чтобы у всех были соответсвующие названия

In [5]:
steam.columns = [
    'link',
    'name',
    'release_date',
    'developers',
    'genre',
    'price',
    'discount',
    'dlc',
    'tags',
    'rating_steam',
    'reviews_count',
    'recent_rating',
    'recent_reviews_count'
]

Объединим датасеты по названию игры:

In [9]:
df = steam.merge(api, on='name', how='inner')
df.head()

,link,name,release_date,developers,genre,price,discount,dlc,tags,rating_steam,reviews_count_x,recent_rating,recent_reviews_count,rating,reviews_count_y,playtime,platforms,stores,esrb_rating,id_games,metacritic,achievements_count,a_count,series_count,dev_team_count
0,https://store.steampowered.com/app/730/Counter...,Counter-Strike 2,2012-08-21,Valve,"['Action', 'Free To Play']",Free To Play,0%,1.0,"['FPS', 'Shooter', 'Multiplayer', 'Competitive...",Very Positive,"9,521,157",Mostly Positive,"94,756",3.57,216.0,0.0,"['PC', 'Linux']",['Steam'],Mature,965470,NaN,0.0,0.0,0.0,1.0
1,https://store.steampowered.com/app/3321460/Cri...,Crimson Desert,NaN,Pearl Abyss,"['Экшены', 'Приключенческие игры']","69,99€",0%,1.0,"['Открытый мир', 'Экшен', 'Для одного игрока',...",Очень положительные,78 700,NaN,NaN,3.68,19.0,0.0,"['PC', 'PlayStation 5', 'Xbox One', 'PlayStati...",NaN,NaN,529829,NaN,0.0,0.0,0.0,0.0
2,https://store.steampowered.com/app/2868840/Sla...,Slay the Spire 2,2026-03-05,Mega Crit,"['Indie', 'Strategy', 'Early Access']","22,99€",0%,0.0,"['Strategy', 'Roguelike', 'Card Game', 'Deckbu...",Very Positive,"98,132",Mostly Positive,"85,557",0.00,5.0,7.0,"['PC', 'macOS', 'Linux']",['Steam'],Everyone 10+,994601,NaN,0.0,0.0,1.0,1.0
3,https://store.steampowered.com/app/2050650/Res...,Resident Evil 4,2023-03-24,"CAPCOM Co., Ltd.","['Action', 'Adventure']","100,24€",-60%,27.0,"['Horror', 'Action', 'Survival Horror', 'Third...",Overwhelmingly Positive,"157,050",Overwhelmingly Positive,"7,371",4.58,635.0,24.0,"['PC', 'PlayStation 5', 'PlayStation 4', 'Xbox...","['Steam', 'PlayStation Store']",Mature,795632,NaN,34.0,0.0,28.0,7.0
4,https://store.steampowered.com/app/1172470/Ape...,Apex Legends™,2020-11-04,Respawn,"['Action', 'Adventure', 'Free To Play']",Free To Play,0%,0.0,"['Free to Play', 'Battle Royale', 'Multiplayer...",Mixed,"1,040,934",Mixed,"8,265",3.63,2439.0,5.0,"['PC', 'Xbox One', 'PlayStation 4', 'Nintendo ...","['Steam', 'PlayStation Store', 'Xbox Store', '...",Teen,290856,80.0,12.0,21.0,1.0,3.0


Узнаем основные данные по получившемуся датафрейму:

In [10]:
print("Размеры датасета: ", df.shape)
print("Колонки датасета: ", df.columns)

Размеры датасета:  (9672, 25)
Колонки датасета:  Index(['link', 'name', 'release_date', 'developers', 'genre', 'price',
       'discount', 'dlc', 'tags', 'rating_steam', 'reviews_count_x',
       'recent_rating', 'recent_reviews_count', 'rating', 'reviews_count_y',
       'playtime', 'platforms', 'stores', 'esrb_rating', 'id_games',
       'metacritic', 'achievements_count', 'a_count', 'series_count',
       'dev_team_count'],
      dtype='object')


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9672 entries, 0 to 9671
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   link                  9672 non-null   object 
 1   name                  9671 non-null   object 
 2   release_date          9665 non-null   object 
 3   developers            9668 non-null   object 
 4   genre                 9672 non-null   object 
 5   price                 9632 non-null   object 
 6   discount              9556 non-null   object 
 7   dlc                   9670 non-null   float64
 8   tags                  9672 non-null   object 
 9   rating_steam          3734 non-null   object 
 10  reviews_count_x       3734 non-null   object 
 11  recent_rating         3569 non-null   object 
 12  recent_reviews_count  3569 non-null   object 
 13  rating                9672 non-null   float64
 14  reviews_count_y       9672 non-null   float64
 15  playtime             

Так как мы хотим предсказать успешность нашей игры, то в качестве целевой переменной можно рассмотреть либо rating_steam, либо rating с сайта RAWG. Видим, что у нас в rating_steam довольно много пропусков, поэтому лучше будет взять признак rating для предсказания

In [11]:
df.isna().sum()

,0
link,0
name,1
release_date,7
developers,4
genre,0
price,40
discount,116
dlc,2
tags,0
rating_steam,5938


In [12]:
#вы скажете гпт а я скажу это копипаст из семинара юры

def missing_values_table(df):
    """
    Функция возвращает резюме по пропущенным значениям
    """
    # Общее число пропусков
    mis_val = df.isnull().sum()

    # Процент пропусков
    mis_val_percent = 100 * df.isnull().sum() / len(df)

    # Создадит таблицу с результатом
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)

    # Переименнуем колонки
    mis_val_table_ren_columns = mis_val_table.rename(
    columns = {0 : 'Missing Values', 1 : '% of Total Values'})

    # Отсортируем по проценту пропущенных значений
    mis_val_table_ren_columns = mis_val_table_ren_columns[
        mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
    '% of Total Values', ascending=False).round(1)

    # Выведем некоторую информацию
    print ("Your selected dataframe has " + str(df.shape[1]) + " columns.\n"
        "There are " + str(mis_val_table_ren_columns.shape[0]) +
            " columns that have missing values.")

    return mis_val_table_ren_columns

In [13]:
missing_values_table(df)

Your selected dataframe has 25 columns.
There are 17 columns that have missing values.


,Missing Values,% of Total Values
metacritic,7090,73.3
esrb_rating,6823,70.5
recent_rating,6103,63.1
recent_reviews_count,6103,63.1
reviews_count_x,5938,61.4
rating_steam,5938,61.4
a_count,1156,12.0
stores,442,4.6
discount,116,1.2
series_count,51,0.5


In [14]:
missing_df = missing_values_table(df)
missing_columns = list(missing_df[missing_df['% of Total Values'] > 50].index)

print('We will remove %d columns.' % len(missing_columns))

Your selected dataframe has 25 columns.
There are 17 columns that have missing values.
We will remove 6 columns.


In [15]:
df = df.drop(columns = list(missing_columns))

In [16]:
df.shape

(9672, 19)

In [17]:
missing_values_table(df)

Your selected dataframe has 19 columns.
There are 11 columns that have missing values.


,Missing Values,% of Total Values
a_count,1156,12.0
stores,442,4.6
discount,116,1.2
series_count,51,0.5
price,40,0.4
platforms,34,0.4
release_date,7,0.1
developers,4,0.0
dev_team_count,3,0.0
dlc,2,0.0


Колонки, где пропусков меньше 1,5%, заполнять не будем. Будем считать, что эти пропуски носят случайный характер и их доля слишком мала, чтобы значимо повлиять на анализ, а заполнение этих пропусков может наоборот внести лишний шум, поэтому удалим строки с этими пропусками

In [18]:
missing_df = missing_values_table(df)
cols_miss = list(missing_df[missing_df['% of Total Values'] <= 1.5].index)
cols_miss

Your selected dataframe has 19 columns.
There are 11 columns that have missing values.


['discount',
 'series_count',
 'price',
 'platforms',
 'release_date',
 'developers',
 'dev_team_count',
 'dlc',
 'name']

In [19]:
df = df.dropna(subset=cols_miss)

In [20]:
df.shape

(9436, 19)

Осталось 2 признака, пропуски в которых необходимо заполнить.

"a_count" - показатель пострелизного развития, те сколько переизданий и всяких продуктов было создано вокруг игры.
Пропуски заменим на 0, так как пропуск как раз и озачает, что доп контент отсутсвет.

"stores" это площадки на которых есть игра, то есть заменим пропуски на пустые скобки, так как по сути получается, что информация о площадках отсуствует.

In [21]:
df["a_count"] = df["a_count"].fillna(0)

In [22]:
df["stores"] = df["stores"].fillna('[]')

In [23]:
df.isna().any().any()

np.False_

Отлично, в итоге у нас не осталось пропусков